# 文件信息提取

In [5]:
import os
import re
import pandas as pd
from datetime import datetime

Data_path = r"E:\\Grad\\data\\中广核响泉风电场"
# Data_path="..//data//中广核响泉风电场"

folder_names = [
    "中广核响泉风电场04#发电机轴承更换后正常数据",
    "中广核响泉风电场04#发电机轴承故障数据",
    "中广核响泉风电场04#主轴承更换后正常数据",
    "中广核响泉风电场04#主轴承故障数据",
    "中广核响泉风电场06#发电机轴承更换后正常数据",
    "中广核响泉风电场06#发电机轴承故障数据",
    "中广核响泉风电场12#发电机轴承更换后正常数据",
    "中广核响泉风电场12#发电机轴承故障数据",
    "中广核响泉风电场14#发电机轴承更换后正常数据",
    "中广核响泉风电场14#发电机轴承故障数据",
    "中广核响泉风电场16#发电机轴承更换后正常数据",
    "中广核响泉风电场16#发电机轴承故障数据",
    "中广核响泉风电场18#发电机轴承更换后数据",
    "中广核响泉风电场18#发电机轴承故障数据",
    "中广核响泉风电场19#发电机轴承更换后数据",
    "中广核响泉风电场19#发电机轴承故障数据",
]

class WindFarmDataReader:
    def __init__(self, base_path):
        self.base_path = base_path
        
    def parse_folder_name_1(self, folder_name):
        """
        解析一级文件夹名称，提取关键信息
        """
        info = {'folder_name': folder_name}  # 保留原始文件夹名
        
        # 提取机组编号
        unit_match = re.search(r'(\d+)#', folder_name)
        info['unit_no'] = unit_match.group(1) if unit_match else '未知'
        
        # 提取部件
        component_match = re.search(r'(发电机轴承|主轴承|齿轮箱)', folder_name)
        info['component'] = component_match.group(1) if component_match else '未知'
        
        # 提取更详细的部件信息（针对特殊情况）
        if '一级行星轮系' in folder_name:
            info['component_detail'] = '一级行星轮系'
        else:
            info['component_detail'] = '无'
        
        # 提取状态
        if '故障' in folder_name:
            info['status'] = '故障'
        elif '正常' in folder_name:
            info['status'] = '正常'
        else:
            info['status'] = '未知'
        
        # 判断是否更换后
        info['after_replacement'] = '更换后' in folder_name
        
        # 构建完整路径
        info['full_path'] = os.path.join(self.base_path, folder_name)
        
        # 检查文件夹是否存在
        info['exists'] = os.path.exists(info['full_path']) and os.path.isdir(info['full_path'])
        
        return info
    
    def create_dataframe(self, folder_names_list):
        """
        将文件夹名称列表转换为DataFrame
        """
        # 解析所有文件夹
        parsed_data = []
        for folder_name in folder_names_list:
            folder_info = self.parse_folder_name_1(folder_name)
            parsed_data.append(folder_info)
        
        # 创建DataFrame
        df = pd.DataFrame(parsed_data)
        
        # 重新排列列的顺序
        column_order = ['folder_name', 'unit_no', 'component', 'component_detail', 
                       'status', 'after_replacement',  ]
        df = df[column_order]
        
        return df
    

Data=WindFarmDataReader(Data_path)
# 创建一级文件夹的DataFrame
Data_info=Data.create_dataframe(folder_names)

Data_info

,folder_name,unit_no,component,component_detail,status,after_replacement
0,中广核响泉风电场04#发电机轴承更换后正常数据,04,发电机轴承,无,正常,True
1,中广核响泉风电场04#发电机轴承故障数据,04,发电机轴承,无,故障,False
2,中广核响泉风电场04#主轴承更换后正常数据,04,主轴承,无,正常,True
3,中广核响泉风电场04#主轴承故障数据,04,主轴承,无,故障,False
4,中广核响泉风电场06#发电机轴承更换后正常数据,06,发电机轴承,无,正常,True
5,中广核响泉风电场06#发电机轴承故障数据,06,发电机轴承,无,故障,False
6,中广核响泉风电场12#发电机轴承更换后正常数据,12,发电机轴承,无,正常,True
7,中广核响泉风电场12#发电机轴承故障数据,12,发电机轴承,无,故障,False
8,中广核响泉风电场14#发电机轴承更换后正常数据,14,发电机轴承,无,正常,True
9,中广核响泉风电场14#发电机轴承故障数据,14,发电机轴承,无,故障,False


In [6]:
import os
import re
import pandas as pd
import numpy as np

# ==========================================
# 1. 辅助函数：标签分类与清洗逻辑
# ==========================================

def clean_component_label(comp_str):
    """
    将 component_detail 转换为标准工程标签
    例如：'非驱动端径向' -> 'NDE_径向'
    """
    if pd.isna(comp_str):
        return "Unknown"

    s = str(comp_str)

    # 定义映射规则
    if "非驱动端" in s:
        prefix = "NDE" # Non-Drive End
    elif "驱动端" in s:
        prefix = "DE"  # Drive End
    else:
        prefix = "Other"

    # 提取方向
    direction = "未知"
    if "径向" in s:
        direction = "径向"
    elif "轴向" in s:
        direction = "轴向"


    return f"{prefix}_{direction}"

def split_status_label(status_str):
    """
    将 status_info 拆分为两列：
    1. run_state: '运行' 或 '停止'
    2. rpm_value: 具体数值 (float)，停止时为 0.0
    """
    if pd.isna(status_str):
        return "未知", np.nan

    s = str(status_str).strip()

    # 判断停止状态
    if "无转速" in s or "0 RPM" in s or "0RPM" in s:
        return "停止", 0.0

    # 判断运行状态并提取数值
    if "RPM" in s:
        # 使用正则提取数字 (支持小数)
        match = re.search(r"(\d+\.?\d*)", s)
        if match:
            rpm_val = float(match.group(1))
            return "运行", rpm_val
        else:
            return "运行", np.nan # 有RPM字样但没读到数

    # 其他情况
    return "未知", np.nan

# ==========================================
# 2. 读取函数 (保持不变，但在返回前可预留接口)
# ==========================================

def read_shiyu(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ 路径不存在：{folder_path}")
        return None

    last_folder = os.path.basename(folder_path)
    print(f"✅ [时域] 开始处理：{last_folder}")

    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    if not files:
        print("   ⚠️ 未找到 .txt 文件")
        return None

    data_list = []
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        try:
            match = re.search(r"(\d{14})", filename)
            if not match: continue
            timestamp_str = match.group(1)
            time_obj = pd.to_datetime(timestamp_str, format='%Y%m%d%H%M%S')

            parts = filename.split('_')
            unit_no = parts[1] if len(parts) > 1 else "Unknown"
            component_detail = parts[3] if len(parts) > 3 else "Unknown"
            status_info = parts[-1].replace('.txt', '') if parts else "Unknown"

            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()

            data_list.append({
                'timestamp': time_obj,
                'turbine_id': unit_no,
                'component_detail': component_detail, # 原始列保留
                'status_info': status_info,           # 原始列保留
                'content': content
            })
        except Exception as e:
            print(f"   读取时域文件 {filename} 出错：{e}")

    if not data_list: return None

    df = pd.DataFrame(data_list).sort_values('timestamp').reset_index(drop=True)
    print(f"   ✅ [时域] 加载完成：{len(df)} 个文件")
    return df

def read_rpm_data(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ 路径不存在：{folder_path}")
        return None

    last_folder = os.path.basename(folder_path)
    print(f"✅ [转速] 开始处理：{last_folder}")

    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    if not files:
        print("   ⚠️ 未找到 .txt 文件")
        return None

    data_list = []
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        try:
            time_match = re.search(r"(\d{14})", filename)
            if not time_match: continue
            timestamp_str = time_match.group(1)
            time_obj = pd.to_datetime(timestamp_str, format='%Y%m%d%H%M%S')

            unit_no = "Unknown"
            if '-' in filename and '_' in filename:
                first_dash = filename.find('-')
                first_under = filename.find('_')
                if first_dash < first_under:
                    unit_no = filename[first_dash+1 : first_under]

            try:
                df_content = pd.read_csv(file_path, header=None, encoding='utf-8')
                rpm_values = df_content.iloc[:, 0].tolist()
                avg_rpm = sum(rpm_values) / len(rpm_values) if rpm_values else 0
            except:
                rpm_values = []
                avg_rpm = None

            data_list.append({
                'timestamp': time_obj,
                'turbine_id': unit_no,
                'monitor_type': '转速 RPM',
                'rpm_avg': avg_rpm,
                'rpm_raw': rpm_values
            })
        except Exception as e:
            print(f"   读取转速文件 {filename} 出错：{e}")

    if not data_list: return None

    df = pd.DataFrame(data_list).sort_values('timestamp').reset_index(drop=True)
    print(f"   ✅ [转速] 加载完成：{len(df)} 个文件")
    return df

# ==========================================
# 3. 主程序：批量执行 + 标签重构
# ==========================================

import os
import pandas as pd

# 配置风机编号列表 (可以添加多个，如 ["04", "05", "06"])
turbine_numbers = ["04", "06", "12", "14","18", "19","27", "28"]
base_dir = r"E:\\Grad\\data\\中广核响泉风电场"

# 用于存储所有风机处理后的干净数据 (列表中包含多个 DataFrame)
all_clean_dfs = []

for t_num in turbine_numbers:
    print(f"\n{'='*60}")
    print(f"🚀 正在处理风机：{t_num}#")
    print(f"{'='*60}")

    # 构建文件夹名称和路径
    folder_name_base = f"中广核响泉风电场{t_num}#发电机轴承更换后正常数据"
    target_shiyu = os.path.join(base_dir, folder_name_base, "时域波形数据")
    target_rpm = os.path.join(base_dir, folder_name_base, "转速")

    # 【新增】检查路径是否存在，避免报错
    if not os.path.exists(target_shiyu):
        print(f"   ⚠️ 警告：找不到路径 '{target_shiyu}'，跳过风机 {t_num}#")
        continue

    try:
        # 1. 读取原始数据
        # 假设 read_shiyu 和 read_rpm_data 是你已经定义好的函数
        df_shiyu = read_shiyu(target_shiyu)
        # df_rpm = read_rpm_data(target_rpm) # 如果后续逻辑需要 rpm 数据，可取消注释

        if df_shiyu is None or df_shiyu.empty:
            print(f"   ⚠️ 警告：风机 {t_num}# 数据为空，跳过。")
            continue

        print("   🔄 正在重构标签 (拆分状态、标准化部件)...")

        # 2. 【核心步骤】对时域数据进行标签重构

        # --- 应用转换函数 ---
        # 拆分 status_info 为两列
        # 注意：确保 split_status_label 返回的是可迭代对象 (如 tuple/list)
        df_shiyu[['run_state', 'rpm_value']] = df_shiyu['status_info'].apply(
            lambda x: pd.Series(split_status_label(x)))

        # 标准化 component_detail
        df_shiyu['sensor_loc_std'] = df_shiyu['component_detail'].apply(
            clean_component_label)

        # 【重要】显式注入当前风机编号，确保合并后能区分来源
        df_shiyu['turbine_id'] = t_num

        # 调整列顺序
        cols_order = [
            'timestamp',
            'turbine_id',  # 现在这一列肯定存在了
            'run_state',
            'rpm_value',  # 新生成的状态列
            'sensor_loc_std',  # 新生成的部件列
            # 'status_info',  # 原始列保留供参考
            'content'
        ]

        # 确保列存在再排序 (防止某些风机原始数据缺少特定列)
        final_cols = [c for c in cols_order if c in df_shiyu.columns]

        # 创建当前风机的干净数据副本 (.copy() 是好习惯，避免 SettingWithCopyWarning)
        df_shiyu_clean = df_shiyu[final_cols].copy()

        # 将当前风机的数据加入总列表
        all_clean_dfs.append(df_shiyu_clean)

        # --- 显示当前风机的结果预览 ---
        preview_cols = [
            c for c in [
                'timestamp', 'turbine_id', 'run_state', 'rpm_value',
                'sensor_loc_std', 'content'
            ] if c in df_shiyu_clean.columns
        ]



        print("\n   📈 [统计信息]:")
        if 'run_state' in df_shiyu_clean.columns:
            print("   - 运行状态分布:")
            print(df_shiyu_clean['run_state'].value_counts())
        if 'sensor_loc_std' in df_shiyu_clean.columns:
            print("   - 传感器位置分布:")
            print(df_shiyu_clean['sensor_loc_std'].value_counts())

        print(f"   ✅ 风机 {t_num}# 处理完成，共 {len(df_shiyu_clean)} 条记录。")
        print("\n   📊 [重构后数据预览] (前 5 行):")
        display(df_shiyu_clean[preview_cols].head())
    except Exception as e:
        print(f"   ❌ 处理风机 {t_num}# 时发生错误：{e}")
        continue

# ==========================================
# 循环结束后的汇总操作
# ==========================================
print(f"\n{'='*60}")
print(f"🎉 批量处理结束！成功处理 {len(all_clean_dfs)} 个风机。")

if len(all_clean_dfs) > 0:
    # 将所有风机的 DataFrame 合并为一个大的 DataFrame
    final_combined_df = pd.concat(all_clean_dfs, ignore_index=True)

    print(f"📦 数据已合并，总行数：{len(final_combined_df)}")
    print(f"🔍 包含的风机编号：{final_combined_df['turbine_id'].unique()}")

    # 这里可以将 final_combined_df 保存到文件或进行下一步分析
    # 示例：final_combined_df.to_csv("all_turbines_clean.csv", index=False, encoding='utf-8-sig')

    # 如果你需要在全局使用这个合并后的表，它可以被赋值给 df_shiyu_clean (或者你喜欢的任何变量名)
    df_shiyu_clean = final_combined_df

    print("\n   📊 [合并后总数据预览] (前 5 行):")
    display(df_shiyu_clean.head())

else:
    print("❌ 没有成功处理任何数据，请检查路径或数据格式。")
    df_shiyu_clean = None


🚀 正在处理风机：04#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
   🔄 正在重构标签 (拆分状态、标准化部件)...

   📈 [统计信息]:
   - 运行状态分布:
run_state
运行    42
停止    30
Name: count, dtype: int64
   - 传感器位置分布:
sensor_loc_std
NDE_径向    24
DE_轴向     24
DE_径向     24
Name: count, dtype: int64
   ✅ 风机 04# 处理完成，共 72 条记录。

   📊 [重构后数据预览] (前 5 行):


,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
0,2023-10-17 02:32:19,04,停止,0.00,NDE_径向,0.9992676\n0.8012695\n0.4205322\n0.5266113\n0....
1,2023-10-17 02:32:19,04,停止,0.00,DE_轴向,-0.3444824\n-0.3818359\n-0.4851074\n0.3560791\...
2,2023-10-17 02:32:19,04,停止,0.00,DE_径向,-1.86731\n0.3044434\n1.657104\n0.6674805\n-0.4...
3,2023-10-17 08:32:21,04,运行,693.57,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...
4,2023-10-17 08:32:21,04,运行,693.57,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....



🚀 正在处理风机：06#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：62 个文件
   🔄 正在重构标签 (拆分状态、标准化部件)...

   📈 [统计信息]:
   - 运行状态分布:
run_state
运行    48
停止    14
Name: count, dtype: int64
   - 传感器位置分布:
sensor_loc_std
NDE_径向    21
DE_径向     21
DE_轴向     20
Name: count, dtype: int64
   ✅ 风机 06# 处理完成，共 62 条记录。

   📊 [重构后数据预览] (前 5 行):


,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
0,2023-12-07 01:06:27,06,运行,728.63,NDE_径向,0.3863525\n-0.5661621\n-0.9152832\n0.02648926\...
1,2023-12-07 01:06:27,06,运行,728.63,DE_径向,-2.753784\n-0.9638672\n-1.029053\n-0.9530029\n...
2,2023-12-07 01:06:27,06,运行,728.63,DE_轴向,-0.03234863\n-0.9841309\n0.7955322\n0.7556152\...
3,2023-12-07 05:06:27,06,运行,446.39,NDE_径向,1.566284\n1.666138\n-0.1895752\n0.2728271\n1.7...
4,2023-12-07 05:06:27,06,运行,446.39,DE_径向,0.7437744\n-0.08996582\n-0.09082031\n1.566406\...



🚀 正在处理风机：12#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
   🔄 正在重构标签 (拆分状态、标准化部件)...

   📈 [统计信息]:
   - 运行状态分布:
run_state
运行    54
停止    18
Name: count, dtype: int64
   - 传感器位置分布:
sensor_loc_std
NDE_径向    24
DE_轴向     24
DE_径向     24
Name: count, dtype: int64
   ✅ 风机 12# 处理完成，共 72 条记录。

   📊 [重构后数据预览] (前 5 行):


...

,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
0,2023-11-13 01:57:54,27,运行,959.05,NDE_径向,-2.297607\n-2.115845\n-1.255493\n-1.366699\n0....
1,2023-11-13 01:57:54,27,运行,959.05,DE_轴向,-1.751465\n-2.071899\n-1.204224\n1.325317\n2.3...
2,2023-11-13 01:57:54,27,运行,959.05,DE_径向,1.257446\n0.8400879\n-0.7473145\n-1.017578\n0....
3,2023-11-13 06:00:52,27,运行,751.39,NDE_径向,1.592651\n1.02063\n-0.5568848\n-2.85437\n-4.88...
4,2023-11-13 06:00:52,27,运行,751.39,DE_轴向,1.633911\n-0.9683838\n-2.001465\n-0.9678955\n-...



🚀 正在处理风机：28#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
   🔄 正在重构标签 (拆分状态、标准化部件)...

   📈 [统计信息]:
   - 运行状态分布:
run_state
运行    63
停止     9
Name: count, dtype: int64
   - 传感器位置分布:
sensor_loc_std
NDE_径向    24
DE_轴向     24
DE_径向     24
Name: count, dtype: int64
   ✅ 风机 28# 处理完成，共 72 条记录。

   📊 [重构后数据预览] (前 5 行):


,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
0,2023-11-13 00:33:22,28,停止,0.00,NDE_径向,1.615723\n4.618408\n8.800293\n-1.283325\n-8.29...
1,2023-11-13 00:33:22,28,停止,0.00,DE_轴向,-1.480591\n-3.151855\n-0.00378418\n0.1307373\n...
2,2023-11-13 00:33:22,28,停止,0.00,DE_径向,2.015381\n-0.9176025\n4.528198\n-1.497437\n4.4...
3,2023-11-13 02:34:52,28,运行,1079.53,NDE_径向,-6.591064\n6.85144\n6.165649\n-3.049438\n0.880...
4,2023-11-13 02:34:52,28,运行,1079.53,DE_轴向,-0.3173828\n0.1062012\n-1.973999\n-2.158569\n-...



🎉 批量处理结束！成功处理 8 个风机。
📦 数据已合并，总行数：557
🔍 包含的风机编号：['04' '06' '12' '14' '18' '19' '27' '28']

   📊 [合并后总数据预览] (前 5 行):


,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
0,2023-10-17 02:32:19,04,停止,0.00,NDE_径向,0.9992676\n0.8012695\n0.4205322\n0.5266113\n0....
1,2023-10-17 02:32:19,04,停止,0.00,DE_轴向,-0.3444824\n-0.3818359\n-0.4851074\n0.3560791\...
2,2023-10-17 02:32:19,04,停止,0.00,DE_径向,-1.86731\n0.3044434\n1.657104\n0.6674805\n-0.4...
3,2023-10-17 08:32:21,04,运行,693.57,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...
4,2023-10-17 08:32:21,04,运行,693.57,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....


In [8]:
with pd.option_context('display.max_rows', None, ):
    
    display(final_combined_df[final_combined_df.turbine_id == "06"])

,timestamp,turbine_id,run_state,rpm_value,sensor_loc_std,content
72,2023-12-07 01:06:27,06,运行,728.63,NDE_径向,0.3863525\n-0.5661621\n-0.9152832\n0.02648926\...
73,2023-12-07 01:06:27,06,运行,728.63,DE_径向,-2.753784\n-0.9638672\n-1.029053\n-0.9530029\n...
74,2023-12-07 01:06:27,06,运行,728.63,DE_轴向,-0.03234863\n-0.9841309\n0.7955322\n0.7556152\...
75,2023-12-07 05:06:27,06,运行,446.39,NDE_径向,1.566284\n1.666138\n-0.1895752\n0.2728271\n1.7...
76,2023-12-07 05:06:27,06,运行,446.39,DE_径向,0.7437744\n-0.08996582\n-0.09082031\n1.566406\...
77,2023-12-07 05:06:27,06,运行,446.39,DE_轴向,0.9434814\n-0.6784668\n-0.1541748\n0.1561279\n...
78,2023-12-07 23:06:34,06,停止,0.00,NDE_径向,-1.157593\n2.101685\n4.12085\n4.812988\n1.7222...
79,2023-12-07 23:06:34,06,停止,0.00,DE_径向,-2.248291\n-1.613281\n-0.4494629\n-2.617188\n-...
80,2023-12-08 13:06:39,06,运行,676.19,NDE_径向,-3.663452\n-2.786255\n-1.758911\n-0.779541\n2....
81,2023-12-08 13:06:39,06,运行,676.19,DE_径向,1.440796\n-0.1433105\n-0.5175781\n-1.280884\n-...


In [ ]:
# 拼接两个数据表
df_shiyu_clean
# 只从 df_rpm 取需要的列
df_rpm_sub = df_rpm[['timestamp', 'turbine_id', 'rpm_raw']]
# 按 timestamp + turbine_id 左连接到 df_shiyu
df_shiyu_clean = df_shiyu_clean.merge(df_rpm_sub,
                                      on=['timestamp', 'turbine_id'],
                                      how='left')
df_shiyu_clean = df_shiyu_clean.fillna(0)
df_shiyu_clean